# mBART — NLP Robot Command Parser

Training, evaluation, ASR and HuRIC evaluation for the **mBART-large-50** model.
Results are saved to `results/mbart/` and checkpoints to `checkpoints/mbart/final`.

Run this notebook independently — it loads data and trains from scratch.

## 0. Setup
Prepares the environment. The repository and the SCAN dataset are cloned from GitHub. Required dependencies are installed from requirements.txt. The configuration file (config.json) is loaded to set model and training parameters.

In [1]:
# Clone repo and move into it
!git clone https://github.com/PetraMicanovic/nlp-robot-command-parser.git
%cd nlp-robot-command-parser

Cloning into 'nlp-robot-command-parser'...
remote: Enumerating objects: 695, done.
remote: Counting objects: 100% (62/62), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 695 (delta 32), reused 44 (delta 15), pack-reused 633 (from 1)
Receiving objects: 100% (695/695), 405.14 KiB | 27.01 MiB/s, done.
Resolving deltas: 100% (390/390), done.
/content/nlp-robot-command-parser


In [2]:
!git clone https://github.com/brendenlake/SCAN.git data/scan

Cloning into 'data/scan'...
remote: Enumerating objects: 205, done.
remote: Total 205 (delta 0), reused 0 (delta 0), pack-reused 205 (from 1)
Receiving objects: 100% (205/205), 11.10 MiB | 19.01 MiB/s, done.
Resolving deltas: 100% (173/173), done.


In [3]:
!pip install -q -r requirements.txt
!pip install -q bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 49.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 128.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
wandb 0.28.0 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.5 MB/s eta 0:00:00


In [4]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import json, sys, torch, random, numpy as np
sys.path.insert(0, '.')   # makes src/ importable

with open('config.json') as f:
    cfg = json.load(f)

SEED = cfg['training']['seed']
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'Model: {cfg["model"]["name"]}')


Device: cuda
Model: t5-small


## 1. Load data
Loads the SCAN dataset using the load_scan function. The data is split into training and test sets based on the configuration.

It can be loaded in either English or Serbian depending on the selected language (`cfg['data']['lang']`). When Serbian is selected, commands and actions are automatically translated.

Basic dataset informations are displayed.

In [ ]:
from src.data.load_data import load_scan

language = cfg['data']['lang'] # 'sr' or 'en'
split = cfg['data']['scan_split']

train_data, test_data = load_scan(
    split = split,
    base_path = cfg['data']['scan_base_path'],
    lang = language,
)

print(f'Language: {language}')
print(f'Train examples: {len(train_data)}')
print(f'Test examples: {len(test_data)}')
print(f'First example: {train_data[0]}')

### 1.1. Dataset statistics

In [ ]:
from src.data.translate_scan import print_stats

print_stats(train_data, test_data)

## 2. Preprocessing (tokenization)

This step prepares the dataset for sequence-to-sequence training with T5. A tokenizer is loaded based on the selected model, and the raw data is converted into Hugging Face `Dataset` format.

The dataset is then tokenized by adding a task-specific prefix, encoding commands and actions, and preparing labels for training. Tokenization is applied separately to the training and test splits using a `DatasetDict`.

In [ ]:
from src.data.preprocess import get_tokenizer, to_hf_dataset, tokenize_dataset
from datasets import DatasetDict

tokenizer = get_tokenizer(cfg['model_mbart']['name'])

raw_dataset = DatasetDict({
    'train': to_hf_dataset(train_data),
    'test': to_hf_dataset(test_data),
})

tokenized_dataset = DatasetDict({
    split: tokenize_dataset(
        raw_dataset[split], tokenizer,
        prefix = cfg['model_mbart']['prefix'],
        max_input_len = cfg['model_mbart']['max_input_len'],
        max_target_len = cfg['model_mbart']['max_target_len'],
    )
    for split in ('train', 'test')
})

print('Tokenization complete.')
print(tokenized_dataset)

## 3. Config for mBART


In [5]:
model_cfg_mbart = cfg["model_mbart"]
training_cfg_mbart = cfg["training_mbart"]

print("Model:", model_cfg_mbart["name"])
print("Out dir:", training_cfg_mbart["output_dir"])

Model: facebook/mbart-large-50
Out dir: checkpoints/mbart


## 4. Training

mBART-large-50 is a multilingual seq2seq model pretrained on 50 languages (including Serbian). Results are saved in a `results/mbart/`.

In [ ]:
#  Load mBART tokeniser + model
from src.models.mbart_model import load_mbart_model

model_mbart, tokenizer_mbart = load_mbart_model(model_cfg_mbart["name"], DEVICE)

In [ ]:
#  Tokenise dataset for mBART
# get_tokenizer detects "mbart" in the name and returns MBart50TokenizerFast
from src.data.preprocess import to_hf_dataset, tokenize_dataset
from datasets import DatasetDict

raw_dataset_mbart = DatasetDict(
    {
        "train": to_hf_dataset(train_data),
        "test": to_hf_dataset(test_data),
    }
)

tokenized_dataset_mbart = tokenize_dataset(
    raw_dataset_mbart,
    tokenizer_mbart,
    prefix=model_cfg_mbart["prefix"],
    max_input_len=model_cfg_mbart["max_input_len"],
    max_target_len=model_cfg_mbart["max_target_len"],
)

In [ ]:
# Train mBART
import gc, torch
from src.training.trainer import build_trainer
from transformers import Seq2SeqTrainingArguments

gc.collect()
torch.cuda.empty_cache()
if DEVICE == "cuda":
    print(f"GPU free memory: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

trainer_mbart = build_trainer(
    model=model_mbart,
    tokenizer=tokenizer_mbart,
    tokenized_dataset=tokenized_dataset_mbart,
    cfg=cfg,
    model_key="model_mbart",
    device_fp16=(DEVICE == "cuda"),
)

trainer_mbart.train()

In [ ]:
from src.training.trainer import get_checkpoint_dir

# Save mBART checkpoint
SAVE_PATH = get_checkpoint_dir(cfg, "model_mbart")
model_mbart.save_pretrained(SAVE_PATH)
tokenizer_mbart.save_pretrained(SAVE_PATH)
print(f"mBART model saved to: {SAVE_PATH}")

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
import shutil, os

drive_dest = f"/content/drive/MyDrive/nlp-robot-command-parser/{SAVE_PATH}"
os.makedirs(drive_dest, exist_ok=True)
shutil.copytree(SAVE_PATH, drive_dest, dirs_exist_ok=True)
print(f"Checkpoint copied to Google Drive: {drive_dest}")

## 5. Loading trained model

In [6]:
from src.training.trainer import get_checkpoint_dir
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast

import os

LOAD_PATH = get_checkpoint_dir(cfg, "model_mbart")

try:
    from google.colab import drive

    drive.mount("/content/drive")
    LOAD_PATH = f"/content/drive/MyDrive/nlp-robot-command-parser/{LOAD_PATH}"
except ImportError:
    pass

if not os.path.exists(LOAD_PATH):
    raise FileNotFoundError(f"Model not found at: {LOAD_PATH}")

model_mbart = MBartForConditionalGeneration.from_pretrained(LOAD_PATH)
tokenizer_mbart = MBart50TokenizerFast.from_pretrained(LOAD_PATH)
model_mbart = model_mbart.to(DEVICE)

print(f"Model loaded from: {LOAD_PATH}")
print(f"Device: {DEVICE}")

Mounted at /content/drive


Loading weights:   0%|          | 0/519 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model loaded from: /content/drive/MyDrive/nlp-robot-command-parser/checkpoints/mbart/final
Device: cuda


## 6. Model Evaluation

This section evaluates the trained mBART model on different SCAN splits.  
The model generates action sequences from input commands and compares them with the correct outputs using exact match accuracy.  

The results are saved for each split.

In [ ]:
from src.models.mbart_model import predict_mbart
from src.evaluation.save_results import save_evaluation_results, copy_results_to_drive
from src.data.load_data import load_scan
import pandas as pd

# SCAN splits used for evaluation
splits_to_eval = ['simple', 'length', 'addprim_jump', 'addprim_turn_left', 'template_around_right','template_jump_around_right','template_opposite_right','template_right','filler_num0','filler_num1','filler_num2','filler_num3','fewshot_num8_rep1']

rows_base = []

# Evaluate model on each SCAN split
for sp in splits_to_eval:
    _, test_sp = load_scan(
        split=sp, base_path=cfg["data"]["scan_base_path"], lang=cfg["data"]["lang"]
    )
    exact = 0
    per_example = []
    for ex in test_sp:
        pred = predict_mbart(
            ex["commands"],
            model_mbart,
            tokenizer_mbart,
            prefix=model_cfg_mbart["prefix"],
            max_input_len=model_cfg_mbart["max_input_len"],
            max_target_len=model_cfg_mbart["max_target_len"],
            device=DEVICE,
            num_beams=model_cfg_mbart["num_beams"],
        ).strip()

        # Ground-truth action sequence
        gold = ex["actions"].strip()
        # Check prediction correctness
        correct = pred == gold
        exact += correct

        # Save per-example results
        per_example.append(
            {
                "command": ex["commands"],
                "gold": gold,
                "predicted": pred,
                "correct": correct,
            }
        )

    results_scan = {
        "exact_match": round(exact / len(test_sp), 4),
        "n_evaluated": len(test_sp),
        "per_example": per_example,
    }

    print("SCAN Evaluation — mBART")
    print("=" * 40)
    print(f'Exact Match: {results_scan["exact_match"]:.2%}')
    print(f"Evaluated: {len(test_sp)} examples")

    save_evaluation_results(
        results_scan, split_name=sp, cfg=cfg, model_key="model_mbart"
    )
copy_results_to_drive(cfg=cfg, model_key="model_mbart")

In [ ]:
import pandas as pd

errors = []
# Collect incorrect predictions
for e in per_example:
    if not e["correct"]:
        errors.append(e)

print(f"Correct: {exact}/{len(test_sp)}")
print(f"Errors: {len(errors)}")

# Display first 10 incorrect predictions
if errors:
    df_errors = pd.DataFrame(errors[:10])
    display(df_errors[["command", "gold", "predicted"]])

## 7. Error analysis by command length

Evaluates model performance for different sequence lengths.
The results are grouped into `buckets`, printed to the console, and saved as a JSON file in the `results` directory.

In [ ]:
from src.evaluation.evaluation import analyse_by_length, print_length_analysis
from src.evaluation.save_results import save_length_analysis, copy_results_to_drive

buckets = analyse_by_length(
    test_data, model_mbart, tokenizer, cfg, DEVICE,
    n=cfg['data']['n_error_analysis']
)
print_length_analysis(buckets)

save_length_analysis(buckets, cfg=cfg, model_key='model_mbart')
copy_results_to_drive(cfg=cfg, model_key='model_mbart')

## 8. End-to-end pipeline evaluation

This section evaluates the complete speech-to-action pipeline.  
Audio commands are first transcribed using Whisper ASR and the generated transcripts are then converted into action sequences using the mBART model.  

The evaluation compares results with and without text normalization and saves the exact match accuracy together with per-example predictions.

In [ ]:
from src.models.asr import transcribe
from src.models.mbart_model import predict_mbart
from src.evaluation.save_results import save_evaluation_results, copy_results_to_drive

# Select random samples for ASR evaluation
asr_sample = random.sample(test_data, cfg["asr"]["n_asr_samples"])

# Paths to normalized and raw audio files
audio_dir_norm = "results/asr_audio"
audio_dir_raw = "results/asr_audio_raw"

# Use Google Drive paths if local paths do not exist
if not os.path.exists(audio_dir_norm):
    audio_dir_norm = "/content/drive/MyDrive/nlp-robot-command-parser/results/asr_audio"
    audio_dir_raw = ("/content/drive/MyDrive/nlp-robot-command-parser/results/asr_audio_raw")

# Counters for exact match accuracy
correct_norm, correct_raw = 0, 0

# Store per-example evaluation results
per_example_asr = []

# Evaluate each ASR sample
for i, sample in enumerate(asr_sample):
    # Ground-truth action sequence
    gold_actions = sample["actions"].strip()

    # Audio file paths
    audio_norm = os.path.join(audio_dir_norm, f"cmd_{i:04d}.mp3")
    audio_raw = os.path.join(audio_dir_raw, f"cmd_raw_{i:04d}.mp3")

    results_i = {}

    for audio_path, normalize, label in [
        (audio_norm, True, "norm"),
        (audio_raw, False, "raw"),
    ]:
        transcript = transcribe(
            audio_path,
            whisper_model_name=cfg["asr"]["whisper_model"],
            language=cfg["asr"]["language"],
            normalize=normalize,
        )
        pred = predict_mbart(
            transcript,
            model_mbart,
            tokenizer_mbart,
            prefix=model_cfg_mbart["prefix"],
            max_input_len=model_cfg_mbart["max_input_len"],
            max_target_len=model_cfg_mbart["max_target_len"],
            device=DEVICE,
            num_beams=model_cfg_mbart["num_beams"],
        ).strip()

        # Check prediction corectness
        correct = pred == gold_actions

        # Update accuracy counters
        if label == "norm":
            correct_norm += correct
        else:
            correct_raw += correct

        # Save intermediate results
        results_i[label] = {
            "transcript": transcript,
            "predicted": pred,
            "correct": correct,
        }

    per_example_asr.append(
        {
            "command": sample["commands"],
            "gold_actions": gold_actions,
            "transcript_norm": results_i["norm"]["transcript"],
            "predicted_norm": results_i["norm"]["predicted"],
            "correct_norm": results_i["norm"]["correct"],
            "transcript_raw": results_i["raw"]["transcript"],
            "predicted_raw": results_i["raw"]["predicted"],
            "correct_raw": results_i["raw"]["correct"],
        }
    )

# Number of evaluated examples
n = len(asr_sample)
pipeline_results = {
    "with_normalization": {"exact_match": round(correct_norm / n, 4), "n_evaluated": n},
    "without_normalization": {
        "exact_match": round(correct_raw / n, 4),
        "n_evaluated": n,
    },
    "per_example": per_example_asr,
}

save_evaluation_results(
    pipeline_results, split_name="pipeline", cfg=cfg, model_key="model_mbart"
)
copy_results_to_drive(cfg=cfg, model_key="model_mbart")

## 8.1. Live voice demo (my own recordings)


In [8]:
from src.models.asr import transcribe
from src.models.mbart_model import predict_mbart
from src.data.translate_scan import translate_actions
from src.evaluation.save_results import save_evaluation_results, copy_results_to_drive
import os
import json
import pandas as pd

voice_dir = 'data/audio/my_voice_demo'
commands_json_path = 'data/my_voice_commands.json'

with open(commands_json_path, 'r', encoding='utf-8') as f:
    voice_commands = json.load(f)

print(f'Loaded {len(voice_commands)} voice command entries from {commands_json_path}')

per_example = []
correct_norm, correct_raw = 0, 0

for entry in voice_commands:
    audio_path = os.path.join(voice_dir, entry['audio_file'])

    if not os.path.exists(audio_path):
        print(f"Missing file: {audio_path}")
        continue

    expected_actions_en = entry.get('output')
    expected_actions_sr = translate_actions(expected_actions_en, cfg['data']['lang'])

    results_i = {}
    for normalize, label in [(True, 'norm'), (False, 'raw')]:
        transcript = transcribe(
            audio_path,
            whisper_model_name=cfg['asr']['whisper_model'],
            language=cfg['asr']['language'],
            normalize=normalize,
        )
        predicted = predict_mbart(
            transcript,
            model_mbart,
            tokenizer_mbart,
            prefix=model_cfg_mbart['prefix'],
            max_input_len=model_cfg_mbart['max_input_len'],
            max_target_len=model_cfg_mbart['max_target_len'],
            device=DEVICE,
            num_beams=model_cfg_mbart['num_beams'],
        ).strip()


        if expected_actions_sr:
          is_correct = (predicted == expected_actions_sr.strip())
        else:
          is_correct ='N/A'
        results_i[label] = {'transcript': transcript, 'predicted': predicted, 'correct': is_correct}

    if results_i['norm']['correct'] is True:
        correct_norm += 1
    if results_i['raw']['correct'] is True:
        correct_raw += 1

    if expected_actions_sr:
        expected_display = expected_actions_sr
    else:
        expected_display = 'N/A'

    per_example.append({
        'File': entry['audio_file'],
        'Command': entry.get('input', ''),
        'Expected actions (sr)': expected_display,
        'Correct': is_correct,
        'Transcript (norm)': results_i['norm']['transcript'],
        'Predicted (norm)': results_i['norm']['predicted'],
        'Correct (norm)': results_i['norm']['correct'],
        'Transcript (raw)': results_i['raw']['transcript'],
        'Predicted (raw)': results_i['raw']['predicted'],
        'Correct (raw)': results_i['raw']['correct'],
    })

n = len(per_example)
if n:
    print(f"\nProcessed {n} recordings")
    print(f"With normalization: {correct_norm}/{n} ({correct_norm/n:.0%})")
    print(f"Without normalization: {correct_raw}/{n} ({correct_raw/n:.0%})")
else:
    print('\nNo recordings processed.')

df_voice_mbart = pd.DataFrame(per_example)
display(df_voice_mbart.style.set_caption('Live voice demo results — mBART'))

if n:
    exact_match_norm = round(correct_norm / n, 4)
    exact_match_raw = round(correct_raw / n, 4)
else:
    exact_match_norm = 0
    exact_match_raw = 0

voice_demo_results = {
    'with_normalization': {'exact_match': exact_match_norm, 'n_evaluated': n},
    'without_normalization': {'exact_match': exact_match_raw, 'n_evaluated': n},
    'per_example': per_example,
}
save_evaluation_results(voice_demo_results, split_name='voice_demo', cfg=cfg, model_key='model_mbart')
copy_results_to_drive(cfg=cfg, model_key='model_mbart')


Loaded 7 voice command entries from data/my_voice_commands.json


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface


Processed 7 recordings
With normalization: 4/7 (57%)
Without normalization: 3/7 (43%)


,File,Command,Expected actions (sr),Correct,Transcript (norm),Predicted (norm),Correct (norm),Transcript (raw),Predicted (raw),Correct (raw)
0,cmd_1.mp3,gledaj desno nakon sto skocis,I_SKOCI I_OKRENI_DESNO I_GLEDAJ,True,gledaj desno nakon sto skocis,I_SKOCI I_OKRENI_DESNO I_GLEDAJ,True,gledaj desno nakon što skočiš.,I_SKOCI I_OKRENI_DESNO I_GLEDAJ,True
1,cmd_2.mp3,hodaj lijevo,I_OKRENI_LIJEVO I_HODAJ,False,hodaj lijevo,I_LIJEVO I_HODAJ,False,hoda i ljevo.,I_HODAJ I_OKRENI_LIJEVO I_HODAJ,False
2,cmd_3.mp3,hodaj lijevo okolo,I_OKRENI_LIJEVO I_HODAJ I_OKRENI_LIJEVO I_HODAJ I_OKRENI_LIJEVO I_HODAJ I_OKRENI_LIJEVO I_HODAJ,False,hodaj lijevo okolo,I_OKRENI_LIJEVO I_HODAJ I_OKRENI_LIJEVO I_HODAJ,False,hoda i lievo okolo.,I_HODAJ I_OKRENI_LIJEVO I_HODAJ I_OKRENI_LIJEVO I_HODAJ I_OKRENI_LIJEVO I_HODAJ,False
3,cmd_4.mp3,hodaj suprotno od desno,I_OKRENI_DESNO I_OKRENI_DESNO I_HODAJ,False,hodaj suprotno od desno,I_DESNO I_OKRENI_DESNO I_HODAJ,False,hoda i suprotno od desno.,I_HODAJ I_OKRENI_DESNO I_OKRENI_DESNO I_HODAJ,False
4,cmd_5.mp3,okreni se desno tri puta i gledaj,I_OKRENI_DESNO I_OKRENI_DESNO I_OKRENI_DESNO I_GLEDAJ,False,pokreni se desno tri puta i gledaj,I_OKRENI_DESNO I_OKRENI_DESNO I_OKRENI_DESNO I_GLEDAJ,True,pokreni se desno 3 puta i gledaj.,I_OKRENI_DESNO I_OKRENI_DESNO I_GLEDAJ,False
5,cmd_6.mp3,skoci tri puta,I_SKOCI I_SKOCI I_SKOCI,True,skoci tri puta,I_SKOCI I_SKOCI I_SKOCI,True,skoči tri puta.,I_SKOCI I_SKOCI I_SKOCI,True
6,cmd_7.mp3,trci dva puta i skoci,I_TRCI I_TRCI I_SKOCI,True,traci dva puta i skoci,I_TRCI I_TRCI I_SKOCI,True,traci dva puta i skoći.,I_TRCI I_TRCI I_SKOCI,True


Evaluation results saved to: results/mbart/evaluation_voice_demo.json
Results folder copied to Google Drive: /content/drive/MyDrive/nlp-robot-command-parser/results/mbart


## 9. HuRIC evaluation

This section evaluates the trained mBART model on the HuRIC dataset.  
Input commands are converted into action sequences and compared with the expected outputs using exact match accuracy.  

The evaluation results and per-example predictions are saved for further analysis.

In [ ]:
import pandas as pd
from src.data.translate_scan import translate_actions
from src.evaluation.save_results import save_evaluation_results, copy_results_to_drive
from src.models.mbart_model import predict_mbart

# Loading HuRIC dataset from JSON file
with open(
    "data/sr_huric_scan_generalization_subset_18.json", "r", encoding="utf-8"
) as f:
    huric_raw = json.load(f)

# Prepare and translate action sequences
huric_test = []
for ex in huric_raw:
    if language == "sr":
        entry = {"commands": ex["input"], "actions": translate_actions(ex["output"], "sr")}
    else:
        entry = {"commands": ex["input"], "actions": translate_actions(ex["output"],),}
    huric_test.append(entry)

print(f"HuRIC examples: {len(huric_test)}")
print(f"Example: {huric_test[0]}")

rows_huric = []
correct_huric = 0

# Evaluation on HuRIC examples
for ex in huric_test:
    # Generate prediction with mBART
    pred = predict_mbart(
        ex["commands"],
        model_mbart,
        tokenizer_mbart,
        prefix=model_cfg_mbart["prefix"],
        max_input_len=model_cfg_mbart["max_input_len"],
        max_target_len=model_cfg_mbart["max_target_len"],
        device=DEVICE,
        num_beams=model_cfg_mbart["num_beams"],
    ).strip()

    # Ground-truth actions
    gold = ex["actions"].strip()

    # Check if prediction matches target
    correct = pred == gold
    correct_huric += correct

    # Save prediction results
    rows_huric.append(
        {
            "Command": ex["commands"],
            "Expected": gold,
            "Predicted": pred,
            "Result": "correct" if correct else "incorrect",
        }
    )

# Compute exact match accuracy
n_huric = len(huric_test)
exact_huric = correct_huric / n_huric

print("HuRIC Evaluation — mBART")
print("=" * 40)
print(f"Exact Match : {exact_huric:.2%}  ({correct_huric}/{n_huric})")

# Create and display results table
df_huric = pd.DataFrame(rows_huric)
df_huric.index += 1
display(df_huric.style.set_caption(f"HuRIC — mBART (Exact match: {exact_huric:.2%})"))

save_evaluation_results(
    {
        "exact_match": round(exact_huric, 4),
        "n_evaluated": n_huric,
        "per_example": rows_huric,
    },
    split_name="huric",
    cfg=cfg,
    model_key="model_mbart",
)
copy_results_to_drive(cfg=cfg, model_key="model_mbart")